# Vision ML System — Cloud Training Pipeline

Run this notebook on **Kaggle** (GPU T4 x2) or **Google Colab** (T4 / A100).

Pipeline:
1. Clone repo + install deps
2. Configure DagsHub credentials (MLflow + DVC remote)
3. Run `prepare_data` stage
4. Run `train` stage (YOLO fine-tuning with MLflow tracking)
5. Run `evaluate` stage → write `metrics/eval_metrics.json`
6. Push trained weights to DagsHub via DVC
7. Show results summary

## 0. Environment check

In [ ]:
import subprocess, sys, os

# Detect environment
ON_KAGGLE = os.path.exists('/kaggle')
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

print(f'Kaggle: {ON_KAGGLE}  |  Colab: {ON_COLAB}')

# GPU check
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('No GPU detected — training will be slow on CPU')

## 1. Clone repository

In [ ]:
REPO_URL   = 'https://github.com/panikeradom/vision-ml-system.git'  # update if different
REPO_DIR   = '/kaggle/working/vision-ml-system' if ON_KAGGLE else '/content/vision-ml-system'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
# DVC with HTTP support for DagsHub remote
!pip install -q 'dvc[http]'kz
print('Dependencies installed.')

## 3. Configure credentials

On **Kaggle**: add `DAGSHUB_USERNAME` and `DAGSHUB_TOKEN` as Kaggle Secrets (Add-ons → Secrets).

On **Colab**: use `userdata.get(...)` or set them manually below.

In [ ]:
import os

if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    DAGSHUB_USERNAME = secrets.get_secret('DAGSHUB_USERNAME')
    DAGSHUB_TOKEN    = secrets.get_secret('DAGSHUB_TOKEN')
elif ON_COLAB:
    from google.colab import userdata
    DAGSHUB_USERNAME = userdata.get('DAGSHUB_USERNAME')
    DAGSHUB_TOKEN    = userdata.get('DAGSHUB_TOKEN')
else:
    # Fallback: set manually (do NOT commit real tokens)
    DAGSHUB_USERNAME = 'panikeradom'
    DAGSHUB_TOKEN    = 'YOUR_TOKEN_HERE'

# Set env vars for MLflow + DVC
os.environ['DAGSHUB_USERNAME']          = DAGSHUB_USERNAME
os.environ['DAGSHUB_TOKEN']             = DAGSHUB_TOKEN
os.environ['MLFLOW_TRACKING_USERNAME']  = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD']  = DAGSHUB_TOKEN
os.environ['MLFLOW_TRACKING_URI']       = f'https://dagshub.com/{DAGSHUB_USERNAME}/vision-ml-system.mlflow'

# Configure DVC remote password (local config, not committed)
!dvc remote modify dagshub --local user {DAGSHUB_USERNAME}
!dvc remote modify dagshub --local password {DAGSHUB_TOKEN}

print('Credentials configured.')

## 4. Training configuration

Override any training params here before running the pipeline.

In [ ]:
import yaml

CONFIG_PATH = 'config/training/base.yaml'

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

# Override for GPU cloud training
config['training']['device']     = 'cuda'   # use GPU
config['training']['epochs']     = 50       # full training run
config['training']['batch_size'] = 32
config['training']['imgsz']      = 640
config['training']['patience']   = 10

# Use coco8 as dataset (built-in, no download needed)
# Replace with your dataset path if you have custom data
config['data']['dataset_yaml']   = None     # None = coco8 fallback in train.py

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('Config updated:')
print(f"  device={config['training']['device']}  epochs={config['training']['epochs']}  "
      f"batch={config['training']['batch_size']}  imgsz={config['training']['imgsz']}")

## 5. Stage 1 — prepare_data

In [ ]:
!python scripts/prepare_data.py --config config/training/base.yaml --source local --output data/prepared

## 6. Stage 2 — train

Trains YOLO11n, logs to MLflow/DagsHub, saves weights to `runs/train/`.

In [ ]:
!python scripts/train.py \
    --config config/training/base.yaml \
    --trigger manual \
    --run-name cloud_gpu_run_v1

## 7. Stage 3 — evaluate

In [ ]:
!python scripts/evaluate.py \
    --config config/training/base.yaml \
    --output metrics/eval_metrics.json

## 8. Results

In [ ]:
import json

metrics_path = 'metrics/eval_metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('\n=== Evaluation Metrics ===')
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f'  {k:<30} {v:.4f}')
        else:
            print(f'  {k:<30} {v}')
else:
    print('metrics/eval_metrics.json not found — check evaluate stage output')

## 9. Push artifacts to DagsHub via DVC

In [ ]:
# Track the trained model with DVC
!dvc add runs/train

# Push to DagsHub remote
!dvc push

print('Artifacts pushed to DagsHub DVC remote.')

## 10. Commit dvc.lock and metrics to git (optional)

Uncomment and run if you want to push the experiment record back to GitHub.

In [ ]:
# !git config user.email "your@email.com"
# !git config user.name "Your Name"
# !git add dvc.lock metrics/eval_metrics.json runs/train.dvc config/training/base.yaml
# !git commit -m "feat: cloud GPU training run — mAP50=$(python -c \"import json; print(json.load(open('metrics/eval_metrics.json'))['mAP50'])\")"  
# !git push

## 11. (Optional) Benchmark backends

After training, benchmark PyTorch FP32 vs FP16 vs ONNX vs TensorRT.

In [ ]:
import glob

# Find best weights from training
weights = sorted(glob.glob('runs/train/**/weights/best.pt', recursive=True),
                 key=os.path.getmtime, reverse=True)
best_pt = weights[0] if weights else 'yolo11n.pt'
print(f'Using weights: {best_pt}')

!python examples/demo_benchmark.py \
    --model {best_pt} \
    --backends pytorch_fp32 pytorch_fp16 onnx tensorrt \
    --frames 200 \
    --warmup 20 \
    --device cuda